# HMR2.0 on contiguous Fit3D clips: does temporal context help when the disc hides the knee?

Runs the 3D-body-prior model (4D-Humans' single-image model, HMR2.0) on EVERY frame of short clips (100 frames at 25 fps,
a 1 s disc over the true knee in the middle, clean context on both sides) and saves its 2D joints for the clean and the disc version.
`src/fit3d/eval_hmr_track.py` then applies temporal filters to the per-frame output and measures the hip-angle error inside the disc block.
This is the video tracker without its detection and appearance parts: on every frame the person is detected, so what a tracker
can add is temporal filtering (or prediction), and this tests that directly. The full PHALP tracker is NOT run here.
Needs a GPU runtime (Runtime > Change runtime type > T4 GPU) and, from you: the SMPL neutral model file and `fit3d_track.zip`
(made by `python src/fit3d/export_clips_tracker.py`). Fit3D must not be redistributed: use the zip only inside your own Colab session.

In [ ]:
import torch, sys
print(torch.__version__, torch.cuda.is_available(), sys.version)
assert torch.cuda.is_available(), "switch the runtime to a GPU"

## 1. Install 4D-Humans (paste any error back)

In [ ]:
!git clone https://github.com/shubham-goel/4D-Humans.git
%cd 4D-Humans
!pip install -q -e .
!pip install -q git+https://github.com/mattloper/chumpy

## 2. SMPL neutral model
Upload `basicModel_neutral_lbs_10_207_0_v1.0.0.pkl` (the same file as last time).

In [ ]:
from google.colab import files
import os, shutil
up = files.upload()   # choose basicModel_neutral_lbs_10_207_0_v1.0.0.pkl
name = list(up)[0]
os.makedirs('data', exist_ok=True)
shutil.copy(name, 'data/basicModel_neutral_lbs_10_207_0_v1.0.0.pkl')
cache = os.path.expanduser('~/.cache/4DHumans/data/smpl')
os.makedirs(cache, exist_ok=True)
shutil.copy(name, os.path.join(cache, 'SMPL_NEUTRAL.pkl'))
print(os.listdir('data'), os.listdir(cache))

## 3. Upload the clips (`outputs/fit3d_track.zip` from your repo folder)

In [ ]:
up = files.upload()   # choose fit3d_track.zip
import zipfile
zipfile.ZipFile(list(up)[0]).extractall('/content/track')
print(sorted(os.listdir('/content/track')))

## 4. Load the model
The first cell lets `torch.load` read the older checkpoint format (the `weights_only` error from last time).

In [ ]:
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
_orig_load = torch.load
torch.load = lambda *a, **k: _orig_load(*a, **{**k, 'weights_only': False})

In [ ]:
import numpy as np, cv2, json
from pathlib import Path
from hmr2.models import download_models, load_hmr2, DEFAULT_CHECKPOINT
from hmr2.utils import recursive_to
from hmr2.datasets.vitdet_dataset import ViTDetDataset

download_models()
model, model_cfg = load_hmr2(DEFAULT_CHECKPOINT)
model = model.cuda().eval()
print('loaded')

## 5. Run on every frame of every clip and save the 2D joints
The person box comes from the true joints (25% padding), as in the single-image run, so the detector is taken out of the comparison.

In [ ]:
def cam_crop_to_full(cam_bbox, box_center, box_size, img_size, focal_length):
    img_w, img_h = img_size[:, 0], img_size[:, 1]
    cx, cy, b = box_center[:, 0], box_center[:, 1], box_size
    bs = b * cam_bbox[:, 0] + 1e-9
    tz = 2 * focal_length / bs
    tx = (2 * (cx - img_w / 2.) / bs) + cam_bbox[:, 1]
    ty = (2 * (cy - img_h / 2.) / bs) + cam_bbox[:, 2]
    return torch.stack([tx, ty, tz], dim=-1)

def joints_2d(img, box):
    ds = ViTDetDataset(model_cfg, img, np.array([box], dtype=np.float32))
    batch = recursive_to(next(iter(torch.utils.data.DataLoader(ds, batch_size=1, shuffle=False))), 'cuda')
    with torch.no_grad():
        out = model(batch)
    isz = batch['img_size'].float()
    f = model_cfg.EXTRA.FOCAL_LENGTH / model_cfg.MODEL.IMAGE_SIZE * isz.max()
    t = cam_crop_to_full(out['pred_cam'], batch['box_center'].float(), batch['box_size'].float(), isz, f)
    X = out['pred_keypoints_3d'][0] + t[0]
    uv = f * X[:, :2] / X[:, 2:3] + isz[0] / 2
    return uv.cpu().numpy()

def read_video(p):
    cap, fr = cv2.VideoCapture(p), []
    while True:
        ok, img = cap.read()
        if not ok:
            break
        fr.append(img)
    return fr

res = {}
for subj in sorted(os.listdir('/content/track')):
    meta = np.load(f'/content/track/{subj}/meta.npz')
    gt = meta['gt2d']
    for cond in ('clean', 'hidden'):
        frames = read_video(f'/content/track/{subj}/{cond}.mp4')
        assert len(frames) == len(gt), (subj, cond, len(frames), len(gt))
        out = []
        for img, g in zip(frames, gt):
            lo, hi = g.min(0), g.max(0)
            pad = 0.25 * (hi - lo)
            box = np.concatenate([np.clip(lo - pad, 0, None), np.minimum(hi + pad, [img.shape[1], img.shape[0]])])
            out.append(joints_2d(img, box))
        res[f'{subj}_{cond}'] = np.array(out)
        print(subj, cond, res[f'{subj}_{cond}'].shape, flush=True)
np.savez('hmr2_track_keypoints.npz', **res)
files.download('hmr2_track_keypoints.npz')

Save the downloaded `hmr2_track_keypoints.npz` into `outputs/fit3d_track/` in the repo, then run `python src/fit3d/eval_hmr_track.py`.